# Step 1: Load the Data and Split It
First, you need to load that massive Parquet file and use the split column (that your teammate brilliantly coded) to carve out your training, validation, and testing sets.

In [2]:
import pandas as pd
import numpy as np
import gc
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# 1. Load the data 
print("Loading data...")
df = pd.read_parquet("data/gkx2020_panel.parquet", engine="fastparquet") 

# 2. Recreate the Time-Series Splits using the 'date' column directly
print("Splitting data...")
train_end = pd.Timestamp("1974-12-31")
valid_end = pd.Timestamp("1986-12-31")
test_end  = pd.Timestamp("2021-12-31")

train_mask = df["date"] <= train_end
valid_mask = (df["date"] > train_end) & (df["date"] <= valid_end)
test_mask  = (df["date"] > valid_end) & (df["date"] <= test_end)

# 3. Define your features (X) and target (y)
# Notice we removed 'split' from cols_to_drop since it's not in the file
cols_to_drop = ['permno', 'date', 'exchcd', 'ret_excess']
feature_cols = [c for c in df.columns if c not in cols_to_drop]

# 4. Apply the masks to create the datasets
print("Assigning X and y...")
X_train, y_train = df.loc[train_mask, feature_cols], df.loc[train_mask, 'ret_excess']
X_valid, y_valid = df.loc[valid_mask, feature_cols], df.loc[valid_mask, 'ret_excess']
X_test,  y_test  = df.loc[test_mask,  feature_cols], df.loc[test_mask,  'ret_excess']

# Free up memory (Crucial for your MacBook!)
del df
gc.collect()

print("Data successfully loaded and split!")

Loading data...
Splitting data...
Assigning X and y...
Data successfully loaded and split!


# Step 2: Train the Random Forest
Because you have over 450,000 rows in the training set and 936 features, a standard Random Forest will take a very long time to train.

**Pro-Tip**: In the code below, max_depth=3 keeps the trees shallow so it runs quickly while you are just testing your code. Once you know it works, you can increase max_depth and tune it using the validation set.

# Step 3: Train the Gradient Boosted Trees (GBRT)
GBRT builds trees sequentially. It is more prone to overfitting than a Random Forest, which is why the learning_rate parameter is critical here.

# Step 4: Saving the Models (Run this right after training)
Add this code in a new cell immediately below where your rf_model.fit() and gbrt_model.fit() code runs.

# Step 5: Loading the Models (When you come back tomorrow)
When you close VS Code and come back later, you do not need to run the .fit() cells again.

Instead, you just need to run your first cell (to load and split the gkx2020_panel.parquet data), and then run this cell to instantly load your trained brains back into memory:

In [ ]:
import joblib

print("Loading saved models from disk...")

# Load the Random Forest
rf_model = joblib.load('models/rf_model_gkx.pkl')

# Load the Gradient Boosted Trees
gbrt_model = joblib.load('models/gbrt_model_gkx.pkl')

print("Models loaded and ready for predictions!")

# Step 6: The Ultimate Test (Calculate $R^2_{OOS}$)
This is the most important calculation for your part of the project. GKX (2020) does not use the standard scikit-learn $R^2$ score. In empirical asset pricing, the benchmark is predicting a zero return.You must calculate the Out-of-Sample R-squared exactly like this on your X_test data:

In [7]:
# 1. Fill missing values with 0 (The median value for cross-sectionally ranked data)
print("Filling missing values with 0...")
X_test = X_test.fillna(0)

# 2. Define the evaluation function
def calculate_oos_r2(y_true, y_pred):
    denominator = (y_true ** 2).sum()
    numerator = ((y_true - y_pred) ** 2).sum()
    r2_oos = 1 - (numerator / denominator)
    return r2_oos

# 3. Generate predictions
print("Generating predictions...")
rf_predictions = rf_model.predict(X_test)
gbrt_predictions = gbrt_model.predict(X_test)

# 4. Calculate and print the results
rf_r2 = calculate_oos_r2(y_test, rf_predictions)
gbrt_r2 = calculate_oos_r2(y_test, gbrt_predictions)

print(f"Random Forest R2_OOS: {rf_r2 * 100:.3f}%")
print(f"Gradient Boosting R2_OOS: {gbrt_r2 * 100:.3f}%")

Filling missing values with 0...
Generating predictions...
Random Forest R2_OOS: -1.115%
Gradient Boosting R2_OOS: -11.825%


# Step 7: Save the Predictions (Run this once)
Right after you generate rf_predictions and gbrt_predictions in your notebook, run this cell to bundle them into a clean dataset and save it to your disk.

In [8]:
import pandas as pd

print("Bundling predictions...")

# 1. Create a DataFrame holding the actual returns and your models' guesses
# Because y_test kept its original index from the main dataset, pandas will line these up perfectly.
predictions_df = pd.DataFrame({
    'actual_return': y_test,
    'rf_predicted': rf_predictions,
    'gbrt_predicted': gbrt_predictions
})

# 2. Save the DataFrame to your data folder
print("Saving predictions to Parquet...")
predictions_df.to_parquet("data/model_predictions.parquet", engine="fastparquet")

print("Predictions successfully saved!")

Bundling predictions...
Saving predictions to Parquet...
Predictions successfully saved!


# Step 8: Load the Predictions (When you come back)
Tomorrow, when you open your laptop to work on Phase 3 (the Portfolio Sorting and economic evaluation), you don't need to load your models or your massive 11 GB dataset.

You just need to run this single cell:

In [ ]:
import pandas as pd

print("Loading saved predictions...")

# Load the predictions directly from your hard drive
results_df = pd.read_parquet("data/model_predictions.parquet", engine="fastparquet")

# Now you can instantly access your numbers!
y_test_loaded = results_df['actual_return']
rf_preds_loaded = results_df['rf_predicted']
gbrt_preds_loaded = results_df['gbrt_predicted']

print(f"Loaded {len(results_df):,} rows of predictions.")